## 面试问题

单步决策粒度：一步产出一个动作还是一组动作？

## 回答主线

本 Notebook 用运营看板 agent 的两类任务对比决策粒度：独立只读的「拉三个指标」适合批量，一步取回；有依赖的「建表→写入→读校验」必须单步，因为写入要以建表成功的回读为前提。验证：(1) 批量执行独立只读动作成功；(2) 把有依赖动作也批量、且顺序被模型排错时，写入先于建表会静默失败、聚合表为空；(3) 改为单步逐个回读后，有依赖任务正确完成。

## 真实案例

看板 agent：任务 A 拉取 `visits/orders/refunds` 三个独立指标；任务 B 是「建临时表 agg → 写入一行 → 计数校验」的依赖链。用一个最小 `TableStore` 模拟外部系统：写入不存在的表会失败。数据为教学假源，不代表真实数仓行为。

In [1]:
metrics_source = {  # 定义独立只读的指标数据源。
    "visits": 1200,  # 当日访问量。
    "orders": 87,  # 当日下单量。
    "refunds": 5,  # 当日退款量。
}  # 结束指标源定义。

def read_metric(name):  # 定义只读指标工具。
    return metrics_source.get(name, None)  # 返回指标值或 None。

print("可用独立指标:", list(metrics_source.keys()))  # 展示可读取的独立指标。

可用独立指标: ['visits', 'orders', 'refunds']


## 基线（Baseline）

对独立只读动作，批量是好基线：一步内并排读取互不依赖的指标，省往返、无风险。

In [2]:
def run_batch_readonly(names):  # 一步内批量执行独立只读动作。
    results = {}  # 收集每个指标的结果。
    for name in names:  # 遍历批次中的每个指标。
        results[name] = read_metric(name)  # 独立读取互不依赖。
    return results  # 返回批量结果。

batch_a = run_batch_readonly(["visits", "orders", "refunds"])  # 批量拉取三个独立指标。
print("批量只读结果:", batch_a)  # 展示独立动作批量执行成功。

批量只读结果: {'visits': 1200, 'orders': 87, 'refunds': 5}


## 失败案例与修正

对有依赖的动作批量执行是危险的。下面模型把批次顺序排成「先写入、再建表、最后计数」，写入时表还不存在——失败被静默吞掉，最终聚合表为空。修正是单步执行并在每步回读前置结果，只有建表成功才写入。

In [3]:
class TableStore:  # 定义一个最小的临时表存储模拟外部系统。
    def __init__(self):  # 初始化存储。
        self.tables = {}  # 用字典保存已建表及其行。
    def create(self, name):  # 建表动作。
        self.tables[name] = []  # 新建一张空表。
        return True  # 返回建表成功。
    def insert(self, name, row):  # 写入动作依赖表已存在。
        if name not in self.tables:  # 表不存在时写入应失败。
            return False  # 返回写入失败。
        self.tables[name].append(row)  # 表存在则追加一行。
        return True  # 返回写入成功。
    def count(self, name):  # 读取校验动作。
        return len(self.tables.get(name, []))  # 返回表行数或 0。

In [4]:
def run_batch_dependent(store):  # 一步内批量执行有依赖的动作而不回读。
    planned = [("insert", "agg", {"k": 1}), ("create", "agg", None), ("count", "agg", None)]  # 模型排出的批次顺序错乱。
    outcomes = []  # 记录每个动作的结果。
    for op in planned:  # 按批次顺序执行且不校验前置。
        if op[0] == "create":  # 建表动作。
            outcomes.append(store.create(op[1]))  # 执行建表。
        elif op[0] == "insert":  # 写入动作。
            outcomes.append(store.insert(op[1], op[2]))  # 执行写入。
        else:  # 读取校验动作。
            outcomes.append(store.count(op[1]))  # 执行计数。
    return outcomes  # 返回批次结果序列。

batch_store = TableStore()  # 新建一个存储实例。
batch_outcomes = run_batch_dependent(batch_store)  # 批量执行有依赖的动作。
print("批量依赖结果:", batch_outcomes, "| 聚合表行数:", batch_store.count("agg"))  # 展示写入先于建表导致失败。

批量依赖结果: [False, True, 0] | 聚合表行数: 0


In [5]:
def run_single_dependent(store):  # 单步执行并在每步回读校验前置。
    log = []  # 记录每步动作与回读。
    created = store.create("agg")  # 第一步建表并回读结果。
    log.append(("create", created))  # 记录建表回读。
    if created:  # 只有建表成功才进入写入。
        inserted = store.insert("agg", {"k": 1})  # 第二步写入一行。
        log.append(("insert", inserted))  # 记录写入回读。
    rows = store.count("agg")  # 第三步读取校验行数。
    log.append(("count", rows))  # 记录校验回读。
    return log  # 返回单步执行日志。

single_store = TableStore()  # 新建独立存储实例。
single_log = run_single_dependent(single_store)  # 单步执行有依赖任务。
print("单步依赖日志:", single_log, "| 聚合表行数:", single_store.count("agg"))  # 展示按依赖顺序回读后成功。

单步依赖日志: [('create', True), ('insert', True), ('count', 1)] | 聚合表行数: 1


## 结果解读

独立只读任务批量执行一次拿到三个指标，没有风险。有依赖任务批量执行时，因为写入先于建表且不回读，`insert` 返回 `False` 被静默忽略，聚合表最终为空——这就是「幻觉式流水线」。改为单步逐个回读后，建表成功才写入，聚合表得到 1 行。粒度选择的本质是：动作间有依赖或副作用时，必须让循环一步一回读。

In [6]:
assert batch_a["visits"] == 1200  # 独立只读动作批量执行应成功。
assert batch_outcomes[0] is False  # 批量里写入先于建表应失败。
assert batch_store.count("agg") == 0  # 乱序批量导致聚合表为空。
assert single_log[0] == ("create", True)  # 单步应先建表并回读成功。
assert single_store.count("agg") == 1  # 单步按依赖顺序写入后应有一行。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
